# PDB Tools

![PDB Tools](https://proto-bio.github.io/proto-assets/images/tool/pdb/hero.png)

This notebook demonstrates the four PDB tools provided by proto_tools. `run_pdb_fetch_entry` retrieves entry-level metadata from the RCSB REST API, returning the structure title, experimental method, and resolution for a given accession. `run_pdb_fetch_fasta` fetches the FASTA sequences for all polymer chains in a structure and automatically classifies each chain as protein or nucleotide based on its composition. `run_pdb_sequence_search` finds polymer entities whose sequences resemble a query, using RCSB's MMseqs2 sequence service ([Steinegger and Söding, 2017](https://doi.org/10.1038/nbt.3988)). `run_pdb_structure_similarity` searches experimental PDB chains or assemblies for structures similar to a query, using RCSB's learned structure embedding ([Segura et al., 2026](https://doi.org/10.1093/bioinformatics/btag058)). Together these functions support common protein design workflows where you need to retrieve a structural template, its sequences, and its structural neighbors before running inverse folding, mutagenesis, or structure prediction.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("pdb")
display_overview("pdb")
display_docs_section("pdb", "Background")

## Available tools

In [ ]:
display_available_tools("pdb")

### `run_pdb_fetch_entry`

`run_pdb_fetch_entry` queries the RCSB REST API to retrieve catalog-level information about a PDB structure: the structure title, experimental method (e.g., X-RAY DIFFRACTION, SOLUTION NMR), and resolution in angstroms. This is useful as a quick lookup step before downloading full structure files, allowing you to confirm the experimental quality and method of a candidate template. We use PDB entry 1EFA, the lac repressor bound to a symmetric operator DNA, as the running example.

In [ ]:
from proto_tools import (
    PdbFetchEntryConfig,
    PdbFetchEntryInput,
    run_pdb_fetch_entry,
)

In [ ]:
# Display docs
display_api_reference("pdb", "input", "run_pdb_fetch_entry")

# Example input
inputs = PdbFetchEntryInput(pdb_id="1EFA")

In [ ]:
# Display config docs
display_api_reference("pdb", "config", "run_pdb_fetch_entry")

# Default config | see docs above for all fields
config = PdbFetchEntryConfig()

In [ ]:
# Run the entry fetch
entry_output = run_pdb_fetch_entry(inputs, config)

In [ ]:
# Display output docs
display_api_reference("pdb", "output", "run_pdb_fetch_entry")

# Inspect the structure metadata
print(f"Title: {entry_output.title}")
print(f"Method: {entry_output.method}")
print(f"Resolution: {entry_output.resolution} A")
print(f"Source: {entry_output.source_url}")

### `run_pdb_fetch_fasta`

`run_pdb_fetch_fasta` retrieves the polymer chain sequences from RCSB PDB in FASTA format and automatically classifies each chain as protein or nucleotide based on its amino acid composition. A PDB structure can contain multiple chains of different types, so this function is useful when you need to extract specific chains for downstream analysis, such as feeding a protein chain into a structure prediction model or aligning it against homologous sequences.

In [ ]:
from proto_tools import (
    PdbFetchFastaConfig,
    PdbFetchFastaInput,
    run_pdb_fetch_fasta,
)

In [ ]:
# Display docs
display_api_reference("pdb", "input", "run_pdb_fetch_fasta")

# Fetch chain sequences for the lac repressor structure
fasta_inputs = PdbFetchFastaInput(pdb_id="1EFA")

In [ ]:
# Display config docs
display_api_reference("pdb", "config", "run_pdb_fetch_fasta")

# Default config | see docs above for all fields
fasta_config = PdbFetchFastaConfig()

In [ ]:
# Run the FASTA fetch
fasta_output = run_pdb_fetch_fasta(fasta_inputs, fasta_config)

In [ ]:
# Display output docs
display_api_reference("pdb", "output", "run_pdb_fetch_fasta")

# Inspect the chain sequences and their classifications
print(f"Chains: {len(fasta_output.chains)}")
print()

for chain in fasta_output.chains:
    chain_type = "protein" if chain.is_protein else "nucleotide"
    preview = chain.sequence[:50] + ("..." if len(chain.sequence) > 50 else "")
    print(f"  Chains {chain.chain_ids}: {chain_type}, length={len(chain.sequence)}")
    print(f"    {preview}")

### `run_pdb_sequence_search`

`run_pdb_sequence_search` aligns each query sequence against every polymer entity in the PDB with RCSB's MMseqs2 sequence service and returns ranked entities with the identity, E-value, bit score, and aligned ranges of the best alignment. By default each hit is also annotated, in one extra request, with the entity description, source organism, UniProt accessions, author chain IDs, and canonical sequence. Because the search tolerates mismatches, it can identify the source of a fragment that a text search would miss. Here we search with a 30-residue fragment of human transthyretin that carries three substitutions.

In [ ]:
from proto_tools import (
    PdbSequenceSearchConfig,
    PdbSequenceSearchInput,
    PdbSequenceSearchQuery,
    run_pdb_sequence_search,
)

In [ ]:
# Display docs
display_api_reference("pdb", "input", "run_pdb_sequence_search")

# A transthyretin fragment with three substitutions (I26V, V32I, K48R)
sequence_inputs = PdbSequenceSearchInput(queries=[PdbSequenceSearchQuery(sequence="VRGSPAVNVAVHIFRKAADDTWEPFASGRT")])

In [ ]:
# Display config docs
display_api_reference("pdb", "config", "run_pdb_sequence_search")

# Keep the 5 best entities at 50% identity or more over the aligned region
sequence_config = PdbSequenceSearchConfig(identity_cutoff=0.5, evalue_cutoff=1.0, max_hits=5)

In [ ]:
# Run the sequence search
sequence_output = run_pdb_sequence_search(sequence_inputs, sequence_config)

In [ ]:
# Display output docs
display_api_reference("pdb", "output", "run_pdb_sequence_search")

result = sequence_output.results[0]
print(f"{result.total_count} matching entities")
for hit in result.hits:
    print(
        f"  {hit.entity_id:8} identity {hit.sequence_identity:.2f}  E={hit.evalue:.1e}  "
        f"chains {','.join(hit.chain_ids or [])}  {hit.uniprot_ids}  {hit.description}"
    )

### `run_pdb_structure_similarity`

`run_pdb_structure_similarity` searches the experimental PDB for chains or assemblies structurally similar to each query. A query is either a public PDB chain or assembly, or a custom `Structure`; custom structures are uploaded to RCSB's public upload service before the search. Each hit carries RCSB's normalized ranking score and a `predicted_tm_score` from the learned embedding, which estimates TM-score without performing an alignment. Several queries can be passed in one call, and results come back in input order. Here we search with both chains of 1YCR: MDM2 (chain A) and the p53 transactivation-domain peptide bound to it (chain B).

In [ ]:
from proto_tools import (
    PdbStructureSimilarityConfig,
    PdbStructureSimilarityInput,
    PdbStructureSimilarityQuery,
    run_pdb_structure_similarity,
)

In [ ]:
# Display docs
display_api_reference("pdb", "input", "run_pdb_structure_similarity")

# Two public queries: MDM2 (chain A) and the bound p53 peptide (chain B)
similarity_inputs = PdbStructureSimilarityInput(
    queries=[
        PdbStructureSimilarityQuery(pdb_id="1YCR", chain_id="A"),
        PdbStructureSimilarityQuery(pdb_id="1YCR", chain_id="B"),
    ]
)

In [ ]:
# Display config docs
display_api_reference("pdb", "config", "run_pdb_structure_similarity")

# Keep the 10 nearest chains with a predicted TM-score of at least 0.5
similarity_config = PdbStructureSimilarityConfig(number_of_candidates=10, predicted_tm_score_cutoff=0.5)

In [ ]:
# Run the structure-similarity search
similarity_output = run_pdb_structure_similarity(similarity_inputs, similarity_config)

In [ ]:
# Display output docs
display_api_reference("pdb", "output", "run_pdb_structure_similarity")

# One result per query, in input order
for query, result in zip(similarity_inputs.queries, similarity_output.results):
    print(f"{query.pdb_id} chain {query.chain_id}: {result.total_count} matches")
    for hit in result.hits[:5]:
        print(f"  {hit.identifier:10} predicted TM-score {hit.predicted_tm_score:.3f}")
    print()

## Export Results

Fetched results can be saved to JSON format for downstream use in other tools or analysis pipelines.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

fasta_output.export("pdb_fasta_results", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'pdb_fasta_results.json'}")